# Country pairwise classification delta

In [ ]:
from __future__ import annotations

import math
import os
from datetime import date
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

from liljegren_analysis import CampaignResults
from liljegren_analysis.country_cells import (
    NATURAL_EARTH_VERSION,
    cached_natural_earth_boundaries,
    country_cells,
    load_countries,
)

In [ ]:
working_directory = Path.cwd().resolve()
repository = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file()
    ),
    None,
)
if repository is None:
    raise FileNotFoundError("Could not locate the repository root")

start_date = date.fromisoformat(os.getenv("SG_ANALYSIS_START_DATE", "2025-01-01"))
end_date = date.fromisoformat(os.getenv("SG_ANALYSIS_END_DATE", "2025-12-31"))
if start_date > end_date:
    raise ValueError("start_date must not be after end_date")


thresholds_c = tuple(
    float(value)
    for value in os.getenv("SG_WBGT_THRESHOLDS_C", "31,32,33").split(",")
)
if (
    not thresholds_c
    or any(not math.isfinite(value) for value in thresholds_c)
    or any(left >= right for left, right in zip(thresholds_c, thresholds_c[1:]))
):
    raise ValueError("thresholds_c must be finite and strictly increasing")

threads = max(1, int(os.getenv("SG_DUCKDB_THREADS", "4")))
output_directory = repository / "data" / "sg_classification"
runs = (
    repository / ".hsb" / "campaigns" / "era5-2025-all-six-m2-w20-v8",
    repository
    / ".hsb"
    / "runs"
    / "era5-2025-meteor-m4-w18"
    / "wbgt-comparison-20260810-231240",
)
missing_runs = [run for run in runs if not (run / "results").is_dir()]
if missing_runs:
    raise FileNotFoundError(f"Missing campaign result roots: {missing_runs}")

{"date_range": (start_date, end_date), "thresholds_c": thresholds_c, "threads": threads}

## Resolve Singapore cells

In [ ]:
boundaries = cached_natural_earth_boundaries(repository / "data" / "boundaries")
singapore = load_countries(boundaries, ["SG"])[0]
selected_cells = country_cells(singapore)
if len(selected_cells) != 4:
    raise ValueError(
        f"Expected 4 Singapore cells under Natural Earth {NATURAL_EARTH_VERSION}, "
        f"found {len(selected_cells)}"
    )

sg_cells = pd.DataFrame(
    [
        {
            "cell_id": cell.cell_id,
            "latitude": cell.latitude,
            "longitude": cell.longitude,
            "center_in_country": cell.center_in_country,
        }
        for cell in selected_cells
    ]
)
cell_ids = tuple(sg_cells["cell_id"])

sg_cells

## Filter in DuckDB, then inspect in pandas

In [ ]:
connection = duckdb.connect()
connection.execute(f"SET threads = {threads}")
try:
    relations = [
        CampaignResults.from_campaign(run, connection=connection).scan(
            start_date=start_date,
            end_date=end_date,
        )
        for run in runs
    ]
    campaign_results = relations[0]
    for relation in relations[1:]:
        campaign_results = campaign_results.union(relation)

    cell_predicate = " OR ".join(
        f"row_id LIKE '{cell_id}-%'" for cell_id in cell_ids
    )
    observations_sgp = (
        campaign_results
        .filter(
            "status = 'ok' AND value_c IS NOT NULL "
            f"AND ({cell_predicate})"
        )
        .project(
            "result_date, run_id, implementation, cycle_ordinal, "
            "row_id, row_ordinal, value_c"
        )
        .df()
    )
finally:
    connection.close()

observations_sgp.info()
display(observations_sgp.head(20))

In [ ]:
identity_columns = [
    "result_date",
    "run_id",
    "implementation",
    "cycle_ordinal",
    "row_id",
]
duplicate_count = int(observations_sgp.duplicated(identity_columns, keep=False).sum())
if duplicate_count:
    raise ValueError(f"observations_sgp contains {duplicate_count} duplicate rows")
if observations_sgp.empty:
    raise ValueError("No valid Singapore observations were found")

observations_sgp["cell_id"] = observations_sgp["row_id"].str[:28]
available_cells = set(observations_sgp["cell_id"].unique())
sg_cells["present_in_campaign"] = sg_cells["cell_id"].isin(available_cells)
observation_summary = pd.DataFrame(
    {
        "valid_observations": [len(observations_sgp)],
        "implementations": [observations_sgp["implementation"].nunique()],
        "cells": [observations_sgp["cell_id"].nunique()],
        "first_date": [observations_sgp["result_date"].min()],
        "last_date": [observations_sgp["result_date"].max()],
    }
)
display(sg_cells)
display(observation_summary)

## Classify and pair

In [ ]:
bounds = (-np.inf, *thresholds_c, np.inf)
class_labels = tuple(
    (
        f"< {upper:g} C"
        if index == 0
        else f">= {lower:g} C"
        if index == len(thresholds_c)
        else f"[{lower:g}, {upper:g}) C"
    )
    for index, (lower, upper) in enumerate(zip(bounds, bounds[1:]))
)
class_definitions = pd.DataFrame(
    {"class_id": range(len(class_labels)), "class_label": class_labels}
)
class_label_by_id = dict(zip(class_definitions["class_id"], class_definitions["class_label"]))

classified_sgp = observations_sgp.copy()
classified_sgp["class_id"] = pd.cut(
    classified_sgp["value_c"],
    bins=bounds,
    labels=False,
    right=False,
).astype("int8")
classified_sgp["class_label"] = classified_sgp["class_id"].map(class_label_by_id)
display(class_definitions)
display(classified_sgp.head(20))

In [ ]:
pair_keys = ["result_date", "run_id", "cycle_ordinal", "row_id", "cell_id"]
pair_values = pair_keys + ["implementation", "value_c", "class_id"]
paired_classifications_sgp = classified_sgp[pair_values].merge(
    classified_sgp[pair_values],
    on=pair_keys,
    how="inner",
    suffixes=("_a", "_b"),
    validate="many_to_many",
)
paired_classifications_sgp = paired_classifications_sgp.loc[
    paired_classifications_sgp["implementation_a"]
    < paired_classifications_sgp["implementation_b"]
].copy()
paired_classifications_sgp["difference_c"] = (
    paired_classifications_sgp["value_c_a"]
    - paired_classifications_sgp["value_c_b"]
)
paired_classifications_sgp["class_difference"] = (
    paired_classifications_sgp["class_id_a"]
    - paired_classifications_sgp["class_id_b"]
)
paired_classifications_sgp["classification_changed"] = (
    paired_classifications_sgp["class_difference"] != 0
)
paired_classifications_sgp.reset_index(drop=True, inplace=True)
paired_classifications_sgp.info()
display(paired_classifications_sgp.head(20))

## Classification-change results

`a_higher_class` means implementation A assigned the higher ordinal class; it does not make A a reference implementation. Transition counts retain direction and form the observed classification confusion matrix for each pair. No agreement or difference-distribution statistics are calculated.

In [ ]:
pair_columns = ["implementation_a", "implementation_b"]
summary_input = paired_classifications_sgp.assign(
    a_higher_class=paired_classifications_sgp["class_difference"] > 0,
    b_higher_class=paired_classifications_sgp["class_difference"] < 0,
)
change_summary = (
    summary_input
    .groupby(pair_columns, as_index=False, sort=True)
    .agg(
        n_paired=("classification_changed", "size"),
        n_changed=("classification_changed", "sum"),
        a_higher_class=("a_higher_class", "sum"),
        b_higher_class=("b_higher_class", "sum"),
    )
)
change_summary["percent_changed"] = (
    100.0 * change_summary["n_changed"] / change_summary["n_paired"]
).round(4)
change_summary = change_summary[
    pair_columns
    + ["n_paired", "n_changed", "percent_changed", "a_higher_class", "b_higher_class"]
]
display(change_summary)

In [ ]:
transition_columns = pair_columns + ["class_id_a", "class_id_b"]
transition_counts = (
    paired_classifications_sgp
    .groupby(transition_columns, as_index=False, sort=True)
    .size()
    .rename(columns={"size": "n"})
)
transition_counts.insert(
    3,
    "class_a_label",
    transition_counts["class_id_a"].map(class_label_by_id),
)
transition_counts.insert(
    5,
    "class_b_label",
    transition_counts["class_id_b"].map(class_label_by_id),
)
transition_counts["classification_changed"] = (
    transition_counts["class_id_a"] != transition_counts["class_id_b"]
)
display(transition_counts)

In [ ]:
# Auditable raw examples; filter paired_classifications_sgp directly for deeper inspection.
changed_observations = (
    paired_classifications_sgp.loc[paired_classifications_sgp["classification_changed"]]
    .assign(absolute_difference_c=lambda frame: frame["difference_c"].abs())
    .sort_values(
        ["absolute_difference_c", "result_date", "row_id"],
        ascending=[False, True, True],
    )
    .drop(columns="absolute_difference_c")
    .head(100)
)
display(changed_observations)

## Export compact classification results

In [ ]:
output_directory.mkdir(parents=True, exist_ok=True)
sg_cells.to_csv(output_directory / "cells.csv", index=False)
class_definitions.to_csv(output_directory / "classes.csv", index=False)
change_summary.to_csv(output_directory / "pair_change_summary.csv", index=False)
transition_counts.to_csv(output_directory / "pair_transition_counts.csv", index=False)
print(f"Saved classification outputs to {output_directory}")